# TMDB Movies Data Pipeline and EDA

In [ ]:
import requests
import pandas as pd
import sqlite3
import json

# --- CONFIG ---
TMDB_API_KEY = 'Your own API'  # Replace with your TMDB API key
TMDB_DISCOVER_URL = 'https://api.themoviedb.org/3/discover/movie'

# --- FETCH MOVIES ---
def fetch_movies(api_key, n_movies):
    params = {
        'api_key': api_key,
        'sort_by': 'popularity.desc',
        'page': 1
    }
    movies = []
    while len(movies) < n_movies:
        response = requests.get(TMDB_DISCOVER_URL, params=params)
        data = response.json()
        movies.extend(data['results'])
        params['page'] += 1
        if params['page'] > data['total_pages']:
            break
    return movies[:n_movies]

# Fetch at least 20 movies
movies = fetch_movies(TMDB_API_KEY, 20)
df_movies = pd.DataFrame(movies)

# --- PREPROCESS LIST/DICT COLUMNS FOR SQLITE ---

for col in df_movies.columns:
    if df_movies[col].apply(lambda x: isinstance(x, (list, dict))).any():
        df_movies[col] = df_movies[col].apply(lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x)

# --- SAVE TO SQLITE ---
conn = sqlite3.connect('movies.db')
df_movies.to_sql('movies', conn, if_exists='replace', index=False)
conn.close()

print('Fetched and saved movies to SQLite.')

Fetched and saved movies to SQLite.


Task2:loading moves table back in to the Data frame
Displaying first 5 Rows

In [ ]:
# --- LOAD FROM SQLITE ---
conn = sqlite3.connect('movies.db')
df = pd.read_sql('SELECT * FROM movies', conn)
conn.close()

# displays the first few rows of the DataFrame 
df.head()

,adult,backdrop_path,genre_ids,id,original_language,original_title,overview,popularity,poster_path,release_date,title,video,vote_average,vote_count
0,0,/1x9e0qWonw634NhIsRdvnneeqvN.jpg,"[10749, 18]",1523145,ru,Твоё сердце будет разбито,High school student Polina is saved from bully...,1165.1266,/7wIBfBl2gejt6xHxNSK0reVIm7E.jpg,2026-03-26,Your Heart Will Be Broken,0,6.827,26
1,0,/u8DU5fkLoM5tTRukzPC31oGPxaQ.jpg,"[878, 12, 14]",83533,en,Avatar: Fire and Ash,In the wake of the devastating war against the...,503.6662,/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg,2025-12-17,Avatar: Fire and Ash,0,7.309,2084
2,0,/uNToXatdunyvWXyXMrTI1nLvh6r.jpg,"[35, 878, 80]",1115544,en,Mike & Nick & Nick & Alice,Two gangsters and the woman they love try to s...,352.9397,/7F0jc75HrSkLVcvOXR2FXAIwuEv.jpg,2026-03-14,Mike & Nick & Nick & Alice,0,6.700,109
3,0,/6GuqUJ33BnWDAoVPZP55gDr5G55.jpg,"[28, 53, 10402]",1084187,en,Pretty Lethal,A troupe of ballerinas find themselves fightin...,311.0726,/znTPnXCK3lEQJgqXCvP7e5FUz6f.jpg,2026-03-13,Pretty Lethal,0,6.745,159
4,0,/nHxWyy18SvAZ8jJeemtS8k1UNjM.jpg,"[28, 80, 53]",1290821,en,Shelter,A man living in self-imposed exile on a remote...,308.3847,/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg,2026-01-28,Shelter,0,6.751,442


In [ ]:
# summary statistics for numeric columns
df.describe()

,adult,id,popularity,video,vote_average,vote_count
count,20.0,2.000000e+01,20.000000,20.0,20.000000,20.000000
mean,0.0,1.062134e+06,263.099260,0.0,6.933350,590.200000
std,0.0,3.097655e+05,234.540901,0.0,0.811891,658.347261
min,0.0,8.353300e+04,101.340400,0.0,5.200000,2.000000
25%,0.0,8.789158e+05,151.561075,0.0,6.479750,146.500000
50%,0.0,1.137552e+06,181.292600,0.0,6.913500,400.000000
75%,0.0,1.271912e+06,300.460725,0.0,7.389000,763.250000
max,0.0,1.523145e+06,1165.126600,0.0,8.500000,2411.000000


In [ ]:
#  number of movies per genre
import collections
import ast

def extract_genres(genre_col):
    genres = []
    for entry in genre_col:
        try:
            genre_list = ast.literal_eval(entry) if isinstance(entry, str) else entry
            if isinstance(genre_list, list):
                genres.extend([g['name'] for g in genre_list if 'name' in g])
        except Exception:
            continue
    return genres

if 'genre_ids' in df.columns:
    print('Note: genre_ids column contains only IDs, not names.')
    genre_counts = df['genre_ids'].explode().value_counts()
    print(genre_counts)
elif 'genres' in df.columns:
    genres_flat = extract_genres(df['genres'])
    genre_counts = collections.Counter(genres_flat)
    print(pd.Series(genre_counts))
else:
    print('No genre information found.')

Note: genre_ids column contains only IDs, not names.
genre_ids
[10749, 18]                  2
[878, 12, 14]                1
[35, 878, 80]                1
[28, 53, 10402]              1
[28, 80, 53]                 1
[16, 35, 10751]              1
[878, 12]                    1
[27, 9648, 80]               1
[27, 53, 35]                 1
[12, 53, 878]                1
[28, 878, 53]                1
[80, 53]                     1
[80, 18]                     1
[18]                         1
[35, 18, 10751, 10749]       1
[16, 10751, 878, 35, 12]     1
[16, 35, 12, 10751, 9648]    1
[27]                         1
[27, 9648]                   1
Name: count, dtype: int64


In [16]:
#columns with missing values
missing = df.isnull().sum()
if missing.any():
    print('Columns with missing values:')
    print(missing)
else:
    print('No missing values found.')

No missing values found.
